Medillian Architecture

In [0]:
# BRONZE: raw ingestion
bronze_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv")

bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce.events")


In [0]:
#silver layer 
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

silver_source = spark.table("ecommerce.events")

window_spec = Window.partitionBy(
    "user_id", "product_id", "event_time"
).orderBy(col("event_time").desc())

silver_df = (
    silver_source
        .withColumn("rn", row_number().over(window_spec))
        .filter(col("rn") == 1)
        .drop("rn")
)

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce.events_table")


In [0]:
#gold layer
from delta.tables import DeltaTable

gold_table = DeltaTable.forName(spark, "ecommerce.events_delta")

silver_df = spark.table("ecommerce.events_table")

gold_table.alias("t").merge(
    silver_df.alias("s"),
    """
    t.user_id = s.user_id AND
    t.product_id = s.product_id AND
    t.event_time = s.event_time
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
#Total Orders per Product
from pyspark.sql.functions import count

gold_orders_df = (
    silver_df
        .filter(col("event_type") == "purchase")
        .groupBy("product_id")
        .agg(
            count("*").alias("total_orders")
        )
)


In [0]:
gold_orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce.gold_product_orders")


In [0]:
display(gold_orders_df)


product_id,total_orders
5701087,20
1005159,2600
8500290,40
17300014,23
6902812,4
15200176,4
7005677,5
28718716,4
9800316,10
12711279,9
